# 确定性策略梯度与 TD3

本章介绍 Deterministic Policy Gradient（DPG，确定性策略梯度）以及 TD3（Twin Delayed Deep Deterministic Policy Gradient）。TD3 由 Fujimoto 等人在 ICML 2018 提出，是连续控制中非常经典的 Actor-Critic 算法。

参考：Fujimoto et al., *Addressing Function Approximation Error in Actor-Critic Methods*, ICML 2018.


## 1. 从随机策略到确定性策略

前面的 Policy Gradient / Actor-Critic 使用随机策略 $\pi_\theta(a\mid s)$。对于连续动作问题，我们也可以直接让 Actor 输出确定性动作：

$$a = \mu_\theta(s).$$

如果 Critic 近似动作价值函数 $Q_\phi(s,a)$，那么 Actor 可以沿着“让 Critic 预测的价值变大”的方向更新。确定性策略梯度写为

$$\nabla_\theta J(\theta) \approx \mathbb{E}_{s\sim \rho}\left[\nabla_a Q_\phi(s,a)|_{a=\mu_\theta(s)} \nabla_\theta \mu_\theta(s)\right].$$

实际实现中可以直接最小化

$$L_{actor}=-\mathbb{E}[Q_\phi(s,\mu_\theta(s))].$$


## 2. Off-policy 与经验回放

TD3 是 off-policy 算法。与环境交互得到的转移 $(s_t,a_t,r_{t+1},s_{t+1})$ 被存入 Replay Buffer（经验回放缓冲区），训练时从历史数据中随机采样 mini-batch。

这样做有两个主要作用：提高样本复用率，并减弱连续时间数据之间的强相关性。


In [ ]:
import numpy as np

class ReplayBuffer:
    def __init__(self, state_dim, action_dim, max_size=100000):
        self.max_size = max_size
        self.state = np.zeros((max_size, state_dim))
        self.action = np.zeros((max_size, action_dim))
        self.next_state = np.zeros((max_size, state_dim))
        self.reward = np.zeros((max_size, 1))
        self.done = np.zeros((max_size, 1))
        self.ptr = 0
        self.size = 0

    def add(self, state, action, next_state, reward, done):
        self.state[self.ptr] = state
        self.action[self.ptr] = action
        self.next_state[self.ptr] = next_state
        self.reward[self.ptr] = reward
        self.done[self.ptr] = float(done)
        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample(self, batch_size):
        idx = np.random.choice(self.size, batch_size, replace=False)
        return (self.state[idx], self.action[idx], self.next_state[idx],
                self.reward[idx], self.done[idx])


## 3. Actor 与 Critic

Actor 输出连续动作，Critic 输入状态与动作并输出一个标量 Q 值。下面给出与教程一致的最小 PyTorch 结构。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.l1 = nn.Linear(state_dim, 256)
        self.l2 = nn.Linear(256, 256)
        self.l3 = nn.Linear(256, action_dim)
        self.max_action = max_action

    def forward(self, state):
        a = F.relu(self.l1(state))
        a = F.relu(self.l2(a))
        return self.max_action * torch.tanh(self.l3(a))

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.l1 = nn.Linear(state_dim + action_dim, 256)
        self.l2 = nn.Linear(256, 256)
        self.l3 = nn.Linear(256, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        x = F.relu(self.l1(x))
        x = F.relu(self.l2(x))
        return self.l3(x)


## 4. 为什么需要 TD3

DDPG 类方法的主要问题之一是 Critic 的函数逼近误差会被 Actor 利用并进一步放大，形成 Q 值高估。TD3 针对这一问题加入了几项关键机制。

### 4.1 Twin Critics

同时学习两个 Critic：$Q_{\phi_1}$ 与 $Q_{\phi_2}$。构造 TD target 时取二者较小值：

$$y=r+\gamma(1-d)\min_{i=1,2} Q_{\phi_i^-}(s',a').$$

这样可以抑制由于近似误差造成的系统性高估。

### 4.2 Target Policy Smoothing

目标动作不是直接使用 $\mu_{\theta^-}(s')$，而是加入裁剪后的高斯噪声：

$$a'=\operatorname{clip}(\mu_{\theta^-}(s')+\epsilon,a_{min},a_{max}).$$

这会使 Critic 学到局部更平滑的价值函数，减少 Actor 利用狭窄 Q 峰值的风险。

### 4.3 Delayed Policy Update

Critic 每一步都更新，而 Actor 与目标网络降低更新频率。直观上先让价值估计更稳定，再更新策略。

### 4.4 Target Network

Actor 和 Critic 都维护慢变化的目标网络，通过 Polyak averaging 更新：

$$\theta^- \leftarrow \tau\theta+(1-\tau)\theta^-.$$

## 5. TD3 的一次训练更新

每次从 Replay Buffer 采样一批数据后：

1. 用目标 Actor 得到下一状态动作，并加入 target policy smoothing noise；
2. 用两个目标 Critic 计算 Q 值并取较小值；
3. 构造 TD target；
4. 更新两个 Critic；
5. 每隔 `policy_freq` 次 Critic 更新，再更新 Actor；
6. 同步软更新 Actor/Critic 的目标网络。


In [ ]:
# TD3 核心 target 计算示意
with torch.no_grad():
    noise = (torch.randn_like(action) * policy_noise).clamp(-noise_clip, noise_clip)
    next_action = (actor_target(next_state) + noise).clamp(min_action, max_action)
    target_q1 = critic1_target(next_state, next_action)
    target_q2 = critic2_target(next_state, next_action)
    target_q = torch.minimum(target_q1, target_q2)
    y = reward + (1.0 - done) * discount * target_q


## 6. 探索噪声与训练初期

确定性 Actor 本身不会产生随机动作，因此训练时通常向 Actor 输出加入 exploration noise。很多实现还会在训练最初一段时间直接随机采样动作，以便 Replay Buffer 中先积累具有多样性的经验。

评估策略时则应关闭探索噪声，使用 Actor 的确定性输出。


## 7. 本章要点

- DPG 直接对确定性 Actor 求梯度，适用于连续动作问题。
- Replay Buffer 让算法成为 off-policy，并显著提高样本复用率。
- Critic 的逼近误差会直接影响 Actor，因此价值高估是核心问题。
- TD3 通过 Twin Critics、Target Policy Smoothing 和 Delayed Policy Update 显著提升稳定性。
- Target Network 与 Polyak averaging 是稳定 bootstrapping 的关键工程机制。

配套练习见 `ex3_td3.ipynb`。
